# 05 Dashboard Prep

This notebook prepares final dashboard-ready tables from the outputs we already created in earlier steps.

## 1. Import libraries and define file paths

We import pandas and define the input files and the folder where the final dashboard-ready tables will be saved.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_OUTPUT_DIR = Path("..") / "outputs"
DASHBOARD_DIR = BASE_OUTPUT_DIR / "dashboard_ready"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_PATH = BASE_OUTPUT_DIR / "events_cleaned.csv"
FUNNEL_OVERALL_PATH = BASE_OUTPUT_DIR / "funnel_overall.csv"
FUNNEL_MONTHLY_PATH = BASE_OUTPUT_DIR / "funnel_monthly.csv"
FUNNEL_CATEGORY_PATH = BASE_OUTPUT_DIR / "funnel_category.csv"
FUNNEL_BRAND_PATH = BASE_OUTPUT_DIR / "funnel_brand.csv"
COHORT_COUNTS_PATH = BASE_OUTPUT_DIR / "cohort_counts.csv"
COHORT_MATRIX_PATH = BASE_OUTPUT_DIR / "cohort_retention_matrix.csv"
COHORT_PERCENTAGES_PATH = BASE_OUTPUT_DIR / "cohort_retention_percentages.csv"

KPI_SUMMARY_OUTPUT = DASHBOARD_DIR / "kpi_summary.csv"
FUNNEL_SUMMARY_OUTPUT = DASHBOARD_DIR / "funnel_summary.csv"
MONTHLY_TRENDS_OUTPUT = DASHBOARD_DIR / "monthly_trends.csv"
CATEGORY_PERFORMANCE_OUTPUT = DASHBOARD_DIR / "category_performance.csv"
BRAND_PERFORMANCE_OUTPUT = DASHBOARD_DIR / "brand_performance.csv"
COHORT_HEATMAP_OUTPUT = DASHBOARD_DIR / "cohort_heatmap.csv"

print(f"Dashboard output folder: {DASHBOARD_DIR.resolve()}")

## 2. Load the existing project outputs

We load the cleaned data and the summary files created in the earlier notebooks.

In [ ]:
events = pd.read_csv(EVENTS_PATH, parse_dates=["event_time"])
funnel_overall = pd.read_csv(FUNNEL_OVERALL_PATH)
funnel_monthly = pd.read_csv(FUNNEL_MONTHLY_PATH)
funnel_category = pd.read_csv(FUNNEL_CATEGORY_PATH)
funnel_brand = pd.read_csv(FUNNEL_BRAND_PATH)
cohort_counts = pd.read_csv(COHORT_COUNTS_PATH)
cohort_matrix = pd.read_csv(COHORT_MATRIX_PATH)
cohort_percentages = pd.read_csv(COHORT_PERCENTAGES_PATH)

print("All input files loaded successfully.")

## 3. Prepare a KPI summary table

This table gives a compact overview of the main project KPIs for a dashboard header or summary cards.

In [ ]:
overall_row = funnel_overall.iloc[0]

kpi_summary = pd.DataFrame(
    {
        "metric_name": [
            "total_events",
            "unique_users",
            "unique_sessions",
            "total_views",
            "total_carts",
            "total_purchases",
            "view_to_cart_rate",
            "cart_to_purchase_rate",
            "view_to_purchase_rate",
            "date_range_start",
            "date_range_end",
        ],
        "metric_value": [
            len(events),
            events["user_id"].nunique(),
            events["user_session"].nunique(),
            int(overall_row["views"]),
            int(overall_row["carts"]),
            int(overall_row["purchases"]),
            round(float(overall_row["view_to_cart_rate"]), 4),
            round(float(overall_row["cart_to_purchase_rate"]), 4),
            round(float(overall_row["view_to_purchase_rate"]), 4),
            events["event_time"].min().strftime("%Y-%m-%d %H:%M:%S"),
            events["event_time"].max().strftime("%Y-%m-%d %H:%M:%S"),
        ],
    }
)

kpi_summary

## 4. Prepare a clean funnel summary table

This table is a tidy version of the overall funnel metrics and is useful for KPI cards or a simple funnel visual.

In [ ]:
funnel_summary = pd.DataFrame(
    {
        "funnel_stage": ["view", "cart", "purchase"],
        "event_count": [
            int(overall_row["views"]),
            int(overall_row["carts"]),
            int(overall_row["purchases"]),
        ],
        "conversion_from_previous_stage": [
            1.0,
            round(float(overall_row["view_to_cart_rate"]), 4),
            round(float(overall_row["cart_to_purchase_rate"]), 4),
        ],
        "conversion_from_view": [
            1.0,
            round(float(overall_row["view_to_cart_rate"]), 4),
            round(float(overall_row["view_to_purchase_rate"]), 4),
        ],
    }
)

funnel_summary

## 5. Prepare monthly trends

This table combines monthly funnel metrics with a few additional monthly activity metrics so it is useful for time-series dashboard charts.

In [ ]:
monthly_activity = (
    events.groupby("event_month")
    .agg(
        total_events=("event_type", "size"),
        unique_users=("user_id", "nunique"),
        unique_sessions=("user_session", "nunique"),
        average_price=("price", "mean"),
    )
    .reset_index()
)

monthly_trends = monthly_activity.merge(funnel_monthly, on="event_month", how="left")
monthly_trends = monthly_trends.rename(columns={
    "views": "total_views",
    "carts": "total_carts",
    "purchases": "total_purchases",
    "view_to_cart_rate": "view_to_cart_conversion_rate",
    "cart_to_purchase_rate": "cart_to_purchase_conversion_rate",
    "view_to_purchase_rate": "view_to_purchase_conversion_rate",
})

monthly_trends["average_price"] = monthly_trends["average_price"].round(2)
monthly_trends = monthly_trends.sort_values("event_month")
monthly_trends

## 6. Prepare category performance

This table keeps the most useful category-level funnel metrics and limits the output to stronger categories for easier dashboarding.

In [ ]:
category_performance = funnel_category.copy()
category_performance = category_performance.rename(columns={
    "category_code": "category_name",
    "views": "total_views",
    "carts": "total_carts",
    "purchases": "total_purchases",
    "view_to_cart_rate": "view_to_cart_conversion_rate",
    "cart_to_purchase_rate": "cart_to_purchase_conversion_rate",
    "view_to_purchase_rate": "view_to_purchase_conversion_rate",
})

category_performance = category_performance[category_performance["total_views"] >= 100].copy()
category_performance = category_performance.sort_values(
    ["view_to_purchase_conversion_rate", "total_purchases"],
    ascending=[False, False],
).head(25)
category_performance

## 7. Prepare brand performance

This table keeps the most useful brand-level funnel metrics and limits the output to stronger brands for easier dashboarding.

In [ ]:
brand_performance = funnel_brand.copy()
brand_performance = brand_performance.rename(columns={
    "brand": "brand_name",
    "views": "total_views",
    "carts": "total_carts",
    "purchases": "total_purchases",
    "view_to_cart_rate": "view_to_cart_conversion_rate",
    "cart_to_purchase_rate": "cart_to_purchase_conversion_rate",
    "view_to_purchase_rate": "view_to_purchase_conversion_rate",
})

brand_performance = brand_performance[brand_performance["total_views"] >= 100].copy()
brand_performance = brand_performance.sort_values(
    ["view_to_purchase_conversion_rate", "total_purchases"],
    ascending=[False, False],
).head(25)
brand_performance

## 8. Prepare the cohort heatmap table

We reshape the retention percentage table into a tidy format so it can be used easily in dashboard tools.

In [ ]:
cohort_heatmap = cohort_percentages.rename(columns={cohort_percentages.columns[0]: "cohort_month"}).copy()
cohort_heatmap = cohort_heatmap.melt(
    id_vars="cohort_month",
    var_name="months_since_first_event",
    value_name="retention_rate",
)
cohort_heatmap["months_since_first_event"] = pd.to_numeric(cohort_heatmap["months_since_first_event"], errors="coerce")
cohort_heatmap = cohort_heatmap.sort_values(["cohort_month", "months_since_first_event"])
cohort_heatmap

## 9. Save the dashboard-ready tables

These are the final files that a dashboard tool can use directly.

In [ ]:
kpi_summary.to_csv(KPI_SUMMARY_OUTPUT, index=False)
funnel_summary.to_csv(FUNNEL_SUMMARY_OUTPUT, index=False)
monthly_trends.to_csv(MONTHLY_TRENDS_OUTPUT, index=False)
category_performance.to_csv(CATEGORY_PERFORMANCE_OUTPUT, index=False)
brand_performance.to_csv(BRAND_PERFORMANCE_OUTPUT, index=False)
cohort_heatmap.to_csv(COHORT_HEATMAP_OUTPUT, index=False)

print(f"Saved: {KPI_SUMMARY_OUTPUT.resolve()}")
print(f"Saved: {FUNNEL_SUMMARY_OUTPUT.resolve()}")
print(f"Saved: {MONTHLY_TRENDS_OUTPUT.resolve()}")
print(f"Saved: {CATEGORY_PERFORMANCE_OUTPUT.resolve()}")
print(f"Saved: {BRAND_PERFORMANCE_OUTPUT.resolve()}")
print(f"Saved: {COHORT_HEATMAP_OUTPUT.resolve()}")

## 10. Print a short summary of the dashboard-ready tables

This final summary explains which files were created and what each one is for.

In [ ]:
table_summary = pd.DataFrame(
    {
        "file_name": [
            "kpi_summary.csv",
            "funnel_summary.csv",
            "monthly_trends.csv",
            "category_performance.csv",
            "brand_performance.csv",
            "cohort_heatmap.csv",
        ],
        "purpose": [
            "High-level KPI cards and summary metrics",
            "Overall funnel counts and conversion rates",
            "Monthly trend charts and time-series analysis",
            "Category-level funnel performance for ranking and comparison",
            "Brand-level funnel performance for ranking and comparison",
            "Cohort retention heatmap source table",
        ],
    }
)

print("Dashboard-ready tables created:")
for _, row in table_summary.iterrows():
    print(f"- {row['file_name']}: {row['purpose']}")

print(f"\nOutput folder: {DASHBOARD_DIR.resolve()}")